# From a PDF to a table: three readers, and a loop

Session 4, Part 4. Five historical PDFs of different kinds, from the Internet Archive:

| | Document | Why it is here |
| --- | --- | --- |
| 1760 | A handwritten report to the Académie des sciences (BIU Santé, Paris) | handwriting |
| 1793 | A printed decree of the département des Vosges (Newberry Library) | old type, the long s (ſ) |
| 1958 | A candidate's leaflet, legislative election (Archelec, Sciences Po) | letterpress, columns |
| 1978 | A candidate's leaflet (Archelec) | offset, photos, slogans |
| 1993 | A candidate's leaflet (Archelec) | modern layout |

Three readers of the same page:

| Reader | What it is | Where it runs | Cost |
| --- | --- | --- | --- |
| **Tesseract** | a classical OCR: shapes, then letters, then words | here, on the CPU | free |
| **DeepSeek-OCR** | an open OCR model (3B parameters), through Ollama | here, on the GPU | free |
| **GPT** (or Claude) | a large model that reads images | OpenAI's servers | a fraction of a cent per page |

The pipeline:

1. **Look** at the PDF, here in the notebook.
2. **Tesseract**: a table of words, each with a position and a confidence.
3. **An LLM**: a transcription, and a small form (type, date, title, people, places).
4. **DeepSeek-OCR**: a free model on this machine.
5. **Compare** the three readings.
6. **Three pages, three readers**: where they part ways. Then **a table** (6b): the hardest page, and how to check it.
7. **A loop**: every page of every document, into one table. That is how it scales.

**Before you start.**
- Runtime, Change runtime type, **T4 GPU**, *before* running anything (DeepSeek-OCR needs it; changing later erases everything).
- Colab Secrets (key icon, left sidebar): `OPENAI_API_KEY`, "Notebook access" on. Never paste a key into a cell. With a Claude key instead: `ANTHROPIC_API_KEY`, and `PROVIDER = "claude"` in Part 0.

The Archelec leaflets are under a restrictive licence (CC BY-NC-ND): look at them, do not republish them.

## 0. Setup (run once; nothing to explain here)

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null
%pip install -q pymupdf pytesseract openai anthropic ollama pydantic

In [ ]:
import base64, datetime, difflib, hashlib, io, json, os, pathlib, re, time, unicodedata
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import pymupdf                      # opens PDFs and turns pages into images
import pytesseract                  # the Python door to Tesseract, the classical OCR
import requests
from IPython.display import display
from PIL import Image, ImageDraw
from pydantic import BaseModel, ConfigDict

# ---- Parameters: everything a referee would ask about is here ----
DOCS = [
    {"id": "manuscript_1760", "kind": "handwriting",
     "url": "https://archive.org/download/BIUSante_ms05541x01/BIUSante_ms05541x01.pdf"},
    {"id": "decree_1793", "kind": "old print, long s",
     "url": "https://archive.org/download/extraitduregistr00vosg_0/extraitduregistr00vosg_0.pdf"},
    {"id": "leaflet_1958", "kind": "letterpress leaflet",
     "url": "https://archive.org/download/EL010_L_1958_11_062_14_1_PF_03/EL010_L_1958_11_062_14_1_PF_03.pdf"},
    {"id": "leaflet_1978", "kind": "offset leaflet",
     "url": "https://archive.org/download/EL105_L_1978_03_073_03_1_PF_03/EL105_L_1978_03_073_03_1_PF_03.pdf"},
    {"id": "leaflet_1993", "kind": "modern leaflet",
     "url": "https://archive.org/download/EL191_L_1993_03_042_06_1_PF_01/EL191_L_1993_03_042_06_1_PF_01.pdf"},
]
MAX_PAGES = 3                  # pages read per document (the first ones)
DPI = 200                      # resolution of the page images given to Tesseract
LANG = "fra"                   # Tesseract's language model
LOW_CONF = 60                  # below this confidence (0 to 100), a word is flagged
PROVIDER = "openai"            # "openai" or "claude": the LLM reader of Part 3
GPT_MODEL = "gpt-6-luna"
CLAUDE_MODEL = "claude-haiku-5-5"
LOCAL_OCR = "deepseek-ocr"     # the free OCR model of Part 4, through Ollama (3B parameters, about 7 GB)
LLM_SIDE = 1568                # longest side, in pixels, of the image sent to the model
PRICE = {"in": 0.10, "out": 0.50}    # $ per million tokens, November 2026 (the same for both models)

WORK = pathlib.Path("s4_ocr_work")
(WORK / "pdf").mkdir(parents=True, exist_ok=True)
BY_ID = {d["id"]: d for d in DOCS}


def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


def fetch_pdf(doc_id):
    """The PDF, as bytes. Downloaded once, then read from disk."""
    f = WORK / "pdf" / f"{doc_id}.pdf"
    if not f.exists():
        for i in range(4):
            r = requests.get(BY_ID[doc_id]["url"], timeout=120, headers={"User-Agent": "uzh-llm-socsci-2026 (course notebook)"})
            if r.status_code == 200:
                f.write_bytes(r.content)
                break
            time.sleep(2 ** i)
        else:
            raise RuntimeError(f"could not download {doc_id}: HTTP {r.status_code}")
    return f.read_bytes()


def page_images(doc_id, dpi=DPI):
    """The first MAX_PAGES pages of a PDF, as images."""
    pdf = pymupdf.open(stream=fetch_pdf(doc_id), filetype="pdf")
    out = []
    for page in list(pdf)[:MAX_PAGES]:
        pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB, alpha=False)
        out.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    return out


def show(img, width=520):
    display(img.resize((width, int(img.height * width / img.width))))


print("ready")

## 1. Look at the PDF

A PDF is a container. Inside: either **text** (a file written on a computer), or **pictures of pages** (a scan). Only the first can be read directly.

In [ ]:
DOC = "leaflet_1958"
pdf = pymupdf.open(stream=fetch_pdf(DOC), filetype="pdf")
len(pdf), [len(page.get_images()) for page in pdf]      # number of pages; pictures on each page

Is there a text layer? Some scans carry the text of an earlier OCR, hidden behind the picture.

In [ ]:
pdf[0].get_text()[:300]

The first page, as a picture. This is what Tesseract and the model will see.

In [ ]:
PAGES = page_images(DOC)
PAGE = PAGES[0]
show(PAGE)

Change `DOC` above to `manuscript_1760` or `decree_1793`, rerun the three cells, and look. Then come back to `leaflet_1958`.

## 2. A classical OCR: Tesseract

Tesseract finds blocks, lines, then words, and recognises each word from its shape. It runs here, on this machine: free, and the same answer every time. What it returns is **a table**: one row per word.

In [ ]:
def tesseract_words(img):
    """One row per word: the text, where it is on the page, and Tesseract's confidence (0 to 100)."""
    t = pytesseract.image_to_data(img, lang=LANG, output_type=pytesseract.Output.DATAFRAME)
    t = t[(t.conf >= 0) & t.text.notna()].copy()
    t["text"] = t.text.astype(str).str.strip()
    return t[t.text != ""][["block_num", "par_num", "line_num", "word_num", "left", "top", "width", "height", "conf", "text"]].reset_index(drop=True)


WORDS = tesseract_words(PAGE)
len(WORDS)

In [ ]:
WORDS.head(12)

Its own doubt: the confidence. The mean over the page, and the words it is least sure of.

In [ ]:
round(WORDS.conf.mean(), 1), round((WORDS.conf < LOW_CONF).mean(), 2)     # mean confidence; share of flagged words

In [ ]:
WORDS.sort_values("conf").head(15)[["conf", "text"]].T

Where are they on the page? In red, the words under `LOW_CONF`.

In [ ]:
def flag(img, words, threshold=LOW_CONF):
    out = img.copy()
    draw = ImageDraw.Draw(out)
    for w in words[words.conf < threshold].itertuples():
        draw.rectangle([w.left, w.top, w.left + w.width, w.top + w.height], outline=(200, 30, 30), width=4)
    return out


show(flag(PAGE, WORDS))

From the table of words back to a text: the words of a line, in order; the lines of a block, in order.

In [ ]:
def text_of(words):
    lines = words.groupby(["block_num", "par_num", "line_num"], sort=True).text.apply(" ".join)
    return "\n".join(lines)


TESS_TEXT = text_of(WORDS)
print(TESS_TEXT[:1200])

**From a text to variables, with code.** A year is easy: four digits. A name is not: "a word with a capital letter" catches the candidate, the town, and the first word of every sentence.

In [ ]:
re.findall(r"\b1[5-9]\d\d\b", TESS_TEXT), re.findall(r"\b[A-ZÉÈ][A-ZÉÈ\-]{3,}\b", TESS_TEXT)[:15]

## 3. An LLM reads the same page

The picture of the page goes to the model with instructions, and a form to fill in. The form has two parts: the **transcription**, and a few **variables** that code alone could not find. GPT by default; Claude if you set `PROVIDER = "claude"`.

In [ ]:
KEY_NAME = {"openai": "OPENAI_API_KEY", "claude": "ANTHROPIC_API_KEY"}[PROVIDER]
assert get_secret(KEY_NAME), f"No key: add {KEY_NAME} in Colab Secrets (key icon), Notebook access on"
if PROVIDER == "openai":
    from openai import OpenAI
    gpt = OpenAI(api_key=get_secret("OPENAI_API_KEY"), max_retries=5)
    LLM_MODEL = GPT_MODEL
else:
    import anthropic
    claude = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"), max_retries=5)
    LLM_MODEL = CLAUDE_MODEL


class PageForm(BaseModel):
    model_config = ConfigDict(extra="forbid")
    transcription: str
    document_type: str
    date: str
    title: str
    people: list[str]
    places: list[str]


INSTRUCTIONS = """You read one page of a scanned historical French document and fill in the form.

transcription: the text of the page, word for word, with its spelling, capitals and accents. Do not correct, do not modernise, do not translate, do not summarise. Write the long s as s. Never complete a word or a sentence you cannot read: write [illisible] in its place. Several columns: one after the other, in reading order. Keep the line breaks of the original, and one blank line between blocks of text. Ignore library stamps and colour charts. An empty page: an empty string.

Then, from this page only, and never from your own knowledge:
document_type: a few words, for example "election leaflet", "administrative decree", "handwritten report". "" if you cannot tell.
date: the date printed or written on the page, as it appears. "" if none.
title: the title or heading of the document, as printed. "" if none.
people: the names of the persons mentioned, as printed. An empty list if none.
places: the towns, départements, and countries mentioned, as printed. An empty list if none."""


def to_jpeg(img, side=LLM_SIDE):
    img = img.copy()
    img.thumbnail((side, side))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=85)
    return base64.standard_b64encode(buf.getvalue()).decode()


def llm_page(img):
    """The page goes to the model as a picture; the form comes back, filled in."""
    if PROVIDER == "openai":
        r = gpt.responses.parse(
            model=GPT_MODEL, max_output_tokens=8000, instructions=INSTRUCTIONS, reasoning={"effort": "none"},
            input=[{"role": "user", "content": [
                {"type": "input_image", "image_url": "data:image/jpeg;base64," + to_jpeg(img), "detail": "high"},
                {"type": "input_text", "text": "Fill in the form for this page."}]}],
            text_format=PageForm)
        form = r.output_parsed.model_dump() if r.output_parsed else None
        return {"form": form, "stop": r.status, "in": r.usage.input_tokens, "out": r.usage.output_tokens}
    r = claude.messages.parse(
        model=CLAUDE_MODEL, max_tokens=8000, system=INSTRUCTIONS, thinking={"type": "disabled"},
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": to_jpeg(img)}},
            {"type": "text", "text": "Fill in the form for this page."}]}],
        output_format=PageForm)
    form = r.parsed_output.model_dump() if r.parsed_output else None
    return {"form": form, "stop": r.stop_reason, "in": r.usage.input_tokens, "out": r.usage.output_tokens}

One page. `stop` should be `completed` (GPT) or `end_turn` (Claude): anything else means the answer was cut.

In [ ]:
ONE = llm_page(PAGE)
ONE["stop"], ONE["in"], ONE["out"]

In [ ]:
LLM_TEXT = ONE["form"]["transcription"]
print(LLM_TEXT[:1200])

And the variables: one row of a table, from one page.

In [ ]:
pd.DataFrame([{k: v for k, v in ONE["form"].items() if k != "transcription"}]).T

## 4. A free OCR model on this machine: DeepSeek-OCR

An open model trained for one job: reading documents. Three billion parameters, about 7 GB, downloaded once and run here through Ollama (Session 3). Free, and the page never leaves the machine. It needs the GPU runtime: on a CPU, count minutes per page.

In [ ]:
# COLAB ONLY: install Ollama in this machine, start it, download the model (3 to 5 minutes). On your laptop: skip.
try:
    import google.colab  # noqa: F401
    import subprocess
    subprocess.run("apt-get install -y -qq zstd pciutils > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(["ollama", "pull", LOCAL_OCR], check=True)
except ImportError:
    print("Not in Colab: Ollama must be running on this machine, with", LOCAL_OCR, "pulled")

In [ ]:
import ollama

try:
    HAS_LOCAL = any(m.model.split(":")[0] == LOCAL_OCR for m in ollama.list().models)
except Exception as e:
    HAS_LOCAL = False
    print("Ollama is not running:", repr(e)[:120])
print(LOCAL_OCR, "ready" if HAS_LOCAL else "not available: Part 4 is skipped, the rest works with two readers")


def deepseek_page(img):
    # The page goes to the local model as a picture; plain text comes back.
    f = WORK / "page.png"
    img.save(f)
    r = ollama.chat(model=LOCAL_OCR, messages=[{"role": "user", "content": "Free OCR.", "images": [str(f)]}],
                    options={"temperature": 0})
    return r.message.content

The same page. The prompt is fixed by the model's authors: `Free OCR.` for plain text.

In [ ]:
t0 = time.time()
DS_TEXT = deepseek_page(PAGE) if HAS_LOCAL else ""
print(f"{time.time() - t0:.1f} seconds\n")
print(DS_TEXT[:1200])

## 5. Compare the three readings

Nobody typed this page by hand: there is no ground truth. What we can do: measure how much the readings differ, look at where, and go back to the picture.

In [ ]:
def norm(s):
    """Lower case, no accents, no punctuation: only the words are compared."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()


def agreement(a, b):
    """Share of words the two texts have in common, in the same order (0 to 1)."""
    a, b = norm(a).split(), norm(b).split()
    return difflib.SequenceMatcher(None, a, b, autojunk=False).ratio() if a or b else 1.0


READINGS = {"tesseract": TESS_TEXT, "llm": LLM_TEXT, "deepseek": DS_TEXT}
if not HAS_LOCAL:
    READINGS.pop("deepseek")
pd.DataFrame({a: {b: round(agreement(READINGS[a], READINGS[b]), 2) for b in READINGS} for a in READINGS})

The words only one reader has. Read the lists with the page in front of you: who is right?

In [ ]:
def only(a, b, k=30):
    return sorted(set(norm(a).split()) - set(norm(b).split()), key=len, reverse=True)[:k]


for name, text in READINGS.items():
    others = " ".join(t for n, t in READINGS.items() if n != name)
    print(f"only {name}:", only(text, others), "\n")

Line by line, where Tesseract and the LLM differ (`-` Tesseract, `+` the LLM).

In [ ]:
def diff(a, b, k=25):
    a = [l.strip() for l in a.splitlines() if l.strip()]
    b = [l.strip() for l in b.splitlines() if l.strip()]
    return [l for l in difflib.unified_diff(a, b, lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")][:k]


diff(TESS_TEXT, LLM_TEXT)

**What to look for.** Tesseract's errors look like errors: broken words, stray signs. The models' errors look like French: a corrected spelling, a completed line, a word that is not on the page. The second kind is harder to see, and worse. DeepSeek-OCR sits in between: a model, but trained only to transcribe.

## 6. Three pages, three readers

Three pages chosen to make the readers part ways: handwriting (1760), old type with the long s (1793), and a leaflet with columns and slogans (1978). For each: the picture, then the three texts side by side.

In [ ]:
import html as _html
from IPython.display import HTML

EXAMPLES = [("manuscript_1760", 0), ("decree_1793", 0), ("leaflet_1978", 0)]


def three_readings(doc_id, page_no):
    img = page_images(doc_id)[page_no]
    out = {"tesseract": text_of(tesseract_words(img))}
    out["llm"] = (llm_page(img)["form"] or {}).get("transcription", "")
    if HAS_LOCAL:
        out["deepseek"] = deepseek_page(img)
    return img, out


def side_by_side(doc_id, page_no, img, texts, chars=900):
    show(img, width=360)
    names = list(texts)
    agree = " | ".join(f"{a} vs {b}: {agreement(texts[a], texts[b]):.2f}"
                       for i, a in enumerate(names) for b in names[i + 1:])
    cols = "".join(f"<td style='vertical-align:top;width:{100 // len(texts)}%'><b>{n}</b>"
                   f"<pre style='white-space:pre-wrap;font-size:11px'>{_html.escape(t[:chars])}</pre></td>"
                   for n, t in texts.items())
    display(HTML(f"<p><b>{doc_id}, page {page_no}</b> &middot; agreement: {agree}</p><table><tr>{cols}</tr></table>"))


SIDE = {}
for doc_id, page_no in EXAMPLES:
    img, texts = three_readings(doc_id, page_no)
    SIDE[(doc_id, page_no)] = texts
    side_by_side(doc_id, page_no, img, texts)

**Read them.** On the manuscript, which reader gives up, and which one invents? On the decree, who reads the long s as an f? On the leaflet, who keeps the columns in order? Write down one place where each reader is the best of the three.

## 6b. A table: the hardest page

A price table of November 1793: the *maximum*, the legal ceiling on prices during the Terror, for the district of Châlons (Newberry Library). Each line: an item, a unit, a price.

For a reader, a table is a trap. The words are easy; the **structure** is not. Tesseract reads across the page, line by line: when a line holds, the item keeps its price; when the dots and the columns confuse it, they part. A model can give back rows and columns, if you ask for them. And then the numbers: they are what you will use, and where an error costs most.

In [ ]:
TABLE_DOC = {"id": "price_table_1793", "kind": "printed price table",
             "url": "https://archive.org/download/secondtableaudum00chlo/secondtableaudum00chlo.pdf"}
BY_ID[TABLE_DOC["id"]] = TABLE_DOC          # not in DOCS: the loop of Part 7 does not read it


def page_image(doc_id, n, dpi=DPI):
    page = pymupdf.open(stream=fetch_pdf(doc_id), filetype="pdf")[n]
    pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB, alpha=False)
    return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)


def contact_sheet(doc_id, dpi=20, cols=7):
    # Every page, small, with its number: to choose the page to read.
    pdf = pymupdf.open(stream=fetch_pdf(doc_id), filetype="pdf")
    ims = []
    for page in pdf:
        pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB, alpha=False)
        ims.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    w, h = max(i.width for i in ims), max(i.height for i in ims)
    rows = (len(ims) + cols - 1) // cols
    sheet = Image.new("RGB", (cols * (w + 8), rows * (h + 24)), "white")
    d = ImageDraw.Draw(sheet)
    for k, im in enumerate(ims):
        x, y = (k % cols) * (w + 8), (k // cols) * (h + 24)
        sheet.paste(im, (x, y + 20))
        d.text((x + 2, y + 2), f"page {k}", fill="black")
    return sheet


display(contact_sheet(TABLE_DOC["id"]))

Choose a page that is mostly table, and look at it closely.

In [ ]:
TABLE_PAGE = 2                 # the page number under the thumbnail
TPAGE = page_image(TABLE_DOC["id"], TABLE_PAGE)
show(TPAGE, width=700)

**Tesseract.** Line by line. Look for an item without its price, and a price on the wrong line.

In [ ]:
TWORDS = tesseract_words(TPAGE)
T_TABLE = text_of(TWORDS)
print(f"{len(TWORDS)} words, mean confidence {TWORDS.conf.mean():.0f}\n")
print(T_TABLE[:1500])

**The LLM, asked for a table.** Not a transcription this time: rows and columns. The form is generic (column names, then one list of cells per row), so the same code works on any table, yours included.

In [ ]:
class Table(BaseModel):
    model_config = ConfigDict(extra="forbid")
    columns: list[str]
    rows: list[list[str]]


TABLE_INSTRUCTIONS = """You read one page of a scanned historical document that contains a table or a price list, and you return it as a table.

columns: the column headings as printed. If the page prints none, name the columns yourself, in a word or two, in square brackets: for example [item], [unit], [price].
rows: one list per line of the table, one cell per column, in the order of the page. Each cell exactly as printed: spelling, abbreviations, numbers, fractions. Do not convert, do not add up, do not correct. A cell you cannot read: [illisible]. An empty cell: "". A heading inside the table: a row with the heading in the first cell and empty cells after it.
Leave out everything that is not part of the table: titles, paragraphs, signatures."""


def llm_table(img):
    """The page goes to the model as a picture; a table comes back."""
    if PROVIDER == "openai":
        r = gpt.responses.parse(
            model=GPT_MODEL, max_output_tokens=16000, instructions=TABLE_INSTRUCTIONS, reasoning={"effort": "none"},
            input=[{"role": "user", "content": [
                {"type": "input_image", "image_url": "data:image/jpeg;base64," + to_jpeg(img, side=2048), "detail": "high"},
                {"type": "input_text", "text": "Return the table of this page."}]}],
            text_format=Table)
        t = r.output_parsed.model_dump() if r.output_parsed else {"columns": [], "rows": []}
        return {"table": t, "stop": r.status, "in": r.usage.input_tokens, "out": r.usage.output_tokens}
    r = claude.messages.parse(
        model=CLAUDE_MODEL, max_tokens=16000, system=TABLE_INSTRUCTIONS, thinking={"type": "disabled"},
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": to_jpeg(img, side=2048)}},
            {"type": "text", "text": "Return the table of this page."}]}],
        output_format=Table)
    t = r.parsed_output.model_dump() if r.parsed_output else {"columns": [], "rows": []}
    return {"table": t, "stop": r.stop_reason, "in": r.usage.input_tokens, "out": r.usage.output_tokens}


def to_frame(t):
    n = max([len(t["columns"])] + [len(r) for r in t["rows"]])
    cols = (t["columns"] + [f"[col {i}]" for i in range(n)])[:n]
    return pd.DataFrame([(r + [""] * n)[:n] for r in t["rows"]], columns=cols)


t0 = time.time()
TAB = llm_table(TPAGE)
FRAME = to_frame(TAB["table"])
print(f"{time.time() - t0:.0f} seconds, stop: {TAB['stop']}, {len(FRAME)} rows, cost ${(TAB['in'] * PRICE['in'] + TAB['out'] * PRICE['out']) / 1e6:.4f}")
FRAME.head(25)

A table you can sort, filter, merge. Save it, with the page it came from:

In [ ]:
FRAME.assign(doc=TABLE_DOC["id"], page=TABLE_PAGE).to_csv(WORK / f"table_{TABLE_DOC['id']}_p{TABLE_PAGE}.csv", index=False)
FRAME[FRAME.apply(lambda r: (r == "[illisible]").any() or (r.iloc[1:] == "").all(), axis=1)]   # headings, and rows with a cell it could not read

**DeepSeek-OCR, asked for a document.** Its second fixed prompt, `Convert the document to markdown.`, keeps the layout: tables come back as tables.

In [ ]:
def deepseek_markdown(img):
    f = WORK / "page.png"
    img.save(f)
    r = ollama.chat(model=LOCAL_OCR, messages=[{"role": "user", "content": "<|grounding|>Convert the document to markdown.",
                                                "images": [str(f)]}], options={"temperature": 0})
    t = re.sub(r"<\|ref\|>.*?<\|/ref\|>|<\|det\|>.*?<\|/det\|>", "", r.message.content, flags=re.S)   # drop the boxes
    return t.strip()


from IPython.display import Markdown

DS_TABLE = deepseek_markdown(TPAGE) if HAS_LOCAL else ""
display(Markdown(DS_TABLE[:5000] or "_DeepSeek-OCR not available_"))

**Check the numbers.** No ground truth again, but numbers are easy to compare: every number each reader found, and the numbers only one of them has. A number only one reader has is a place to look, with the picture open.

In [ ]:
from collections import Counter


def numbers(text):
    return Counter(re.findall(r"\d+", str(text)))


NUMS = {"tesseract": numbers(T_TABLE),
        "llm": numbers(" ".join(" ".join(r) for r in TAB["table"]["rows"])),
        "deepseek": numbers(DS_TABLE)}
if not HAS_LOCAL:
    NUMS.pop("deepseek")
for name, c in NUMS.items():
    rest = sum((v for k, v in NUMS.items() if k != name), Counter())
    print(f"{name}: {sum(c.values())} numbers; only {name}: {sorted((c - rest).elements(), key=int)[:40]}")

**What to look for.** The text is the easy part; the structure is the hard part. Tesseract keeps the words but not the columns. The models give rows you can merge, but a price can slip to the next line, and nothing in the table will tell you: the table looks clean either way. Check a sample of rows against the picture, and write down how many were wrong.

On your own tables: crop the table out of the page before you send it, cut a dense table into strips of rows, and ask for the cells exactly as printed. Totals, when the page has them, are a free check: add the column up yourself.

## 7. The loop: every page, one table

One function does everything for one page. The loop calls it for every page of every document. Each answer is written to a file as it arrives: a rerun does not pay twice, and a crash loses nothing. On 5 documents or 5,000, the code is the same.

In [ ]:
CONFIG = hashlib.sha256((INSTRUCTIONS + LLM_MODEL + str(DPI) + LANG + (LOCAL_OCR if HAS_LOCAL else "")).encode()).hexdigest()[:8]
RAW = WORK / f"pages_{CONFIG}.jsonl"          # every answer, kept


def process(task):
    doc_id, page_no, img = task
    t0 = time.time()
    words = tesseract_words(img)
    t1 = time.time()
    try:
        llm = llm_page(img)
    except Exception as e:
        llm = {"form": None, "error": repr(e)[:300]}
    t2 = time.time()
    ds = deepseek_page(img) if HAS_LOCAL else ""
    return {"doc_id": doc_id, "page": page_no, "tess_text": text_of(words), "tess_words": len(words),
            "tess_conf": round(float(words.conf.mean()), 1) if len(words) else None, "ds_text": ds,
            "tess_seconds": round(t1 - t0, 1), "llm_seconds": round(t2 - t1, 1),
            "ds_seconds": round(time.time() - t2, 1), **llm}


DONE = {}
if RAW.exists():
    for line in RAW.read_text().splitlines():
        rec = json.loads(line)
        if rec.get("form"):
            DONE[(rec["doc_id"], rec["page"])] = rec

TASKS = [(d["id"], n, img) for d in DOCS for n, img in enumerate(page_images(d["id"])) if (d["id"], n) not in DONE]
print(len(DONE), "pages already done,", len(TASKS), "to do")
with ThreadPoolExecutor(1 if HAS_LOCAL else 4) as pool, RAW.open("a") as f:   # one GPU: one page at a time
    for rec in pool.map(process, TASKS):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        DONE[(rec["doc_id"], rec["page"])] = rec
        print(" ", rec["doc_id"], rec["page"], "ok" if rec.get("form") else rec.get("error"))

The table: one row per page. Tesseract's confidence, how much each pair of readers agrees, then the variables the LLM extracted.

In [ ]:
def row(rec):
    f = rec.get("form") or {}
    llm_t, ds_t = f.get("transcription", ""), rec.get("ds_text", "")
    return {"doc_id": rec["doc_id"], "page": rec["page"], "kind": BY_ID[rec["doc_id"]]["kind"],
            "tess_words": rec["tess_words"], "tess_conf": rec["tess_conf"],
            "agreement": round(agreement(rec["tess_text"], llm_t), 2),
            "tess_vs_ds": round(agreement(rec["tess_text"], ds_t), 2) if ds_t else None,
            "llm_vs_ds": round(agreement(llm_t, ds_t), 2) if ds_t else None,
            "llm_words": len(f.get("transcription", "").split()), "illisible": f.get("transcription", "").count("[illisible]"),
            "type": f.get("document_type"), "date": f.get("date"), "title": (f.get("title") or "")[:60],
            "people": "; ".join(f.get("people", []))[:80], "places": "; ".join(f.get("places", []))[:60]}


order = {d["id"]: k for k, d in enumerate(DOCS)}
TABLE = pd.DataFrame([row(r) for r in sorted(DONE.values(), key=lambda r: (order.get(r["doc_id"], 99), r["page"]))])
pd.set_option("display.max_colwidth", 80)
TABLE

**Read the table by kind of document.** Where is Tesseract confident, and where does it give up? Which pairs of readers agree? Two readers that agree can both be wrong in the same way; a reader that returns almost nothing (Tesseract on handwriting) says nothing about the others. Only your eyes settle it.

In [ ]:
TABLE.groupby("kind", sort=False)[["tess_conf", "agreement", "tess_vs_ds", "llm_vs_ds", "tess_words", "llm_words"]].mean().round(2)

One page where they disagree most: the picture, then both texts.

In [ ]:
worst = TABLE.sort_values("agreement").iloc[0]
rec = DONE[(worst.doc_id, int(worst.page))]
print(worst.doc_id, "page", worst.page, "| agreement", worst.agreement, "| Tesseract confidence", worst.tess_conf)
show(page_images(worst.doc_id)[int(worst.page)])

In [ ]:
print("TESSERACT\n", rec["tess_text"][:700], "\n\nTHE LLM\n", (rec["form"] or {}).get("transcription", "")[:700],
      "\n\nDEEPSEEK-OCR\n", rec.get("ds_text", "")[:700])

### What it costs, and how it scales

Time and money per page, from this run. Then the same loop on a real corpus: the 4,010 leaflets of the 1978 election have about two pages each.

In [ ]:
recs = [r for r in DONE.values() if r.get("form")]
cost = sum(r["in"] * PRICE["in"] + r["out"] * PRICE["out"] for r in recs) / 1e6
per_page = {"tesseract seconds": round(sum(r["tess_seconds"] for r in recs) / len(recs), 1),
            "deepseek seconds": round(sum(r.get("ds_seconds", 0) for r in recs) / len(recs), 1),
            "llm seconds": round(sum(r["llm_seconds"] for r in recs) / len(recs), 1),
            "llm dollars": round(cost / len(recs), 4)}
print(per_page)
print(f"8,000 pages: Tesseract about {per_page['tesseract seconds'] * 8000 / 3600:.1f} hours on one core, free; "
      f"DeepSeek-OCR about {per_page['deepseek seconds'] * 8000 / 3600:.1f} hours on this GPU, free; "
      f"the LLM about ${per_page['llm dollars'] * 8000:.0f}")

**At scale, three things change, not the code.**

- **Tesseract**: one core per page. On the ScienceCluster, a job with 32 cores reads 32 pages at a time (`sessions/04/cluster/`).
- **DeepSeek-OCR**: one GPU, one page at a time; on the cluster, one job per GPU, as many GPUs as the queue gives you.
- **The LLM**: the batch API takes all the requests at once, at half price, and answers within a day (Session 3).
- **You**: you cannot read 8,000 pages. You read a random sample, and you let the two readings point at the pages where they disagree.

Keep what a referee would ask for: the table, every raw answer, the instructions, the model, the date.

In [ ]:
TABLE.to_csv(WORK / "table.csv", index=False)
(WORK / "params.json").write_text(json.dumps({
    "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "documents": DOCS, "max_pages": MAX_PAGES,
    "tesseract": {"version": str(pytesseract.get_tesseract_version()), "lang": LANG, "dpi": DPI},
    "llm": {"provider": PROVIDER, "model": LLM_MODEL, "image_side": LLM_SIDE, "instructions_sha256": hashlib.sha256(INSTRUCTIONS.encode()).hexdigest()},
    "local_ocr": {"model": LOCAL_OCR, "runtime": "Ollama", "prompt": "Free OCR."} if HAS_LOCAL else "not used",
    "raw_answers": RAW.name}, indent=2, ensure_ascii=False))
sorted(p.name for p in WORK.iterdir())

## 8. Your own PDF

Add a line to `DOCS` with the address of a PDF, or upload one (Colab: Files panel) and give its path. Then rerun Part 7: the loop only reads what is new.

In [ ]:
MY_PDF = ""          # a URL, or the path of a file you uploaded, e.g. "my_scan.pdf"
if MY_PDF:
    doc = {"id": pathlib.Path(MY_PDF).stem, "kind": "mine", "url": MY_PDF}
    if not MY_PDF.startswith("http"):
        (WORK / "pdf" / f"{doc['id']}.pdf").write_bytes(pathlib.Path(MY_PDF).read_bytes())
    if doc["id"] not in BY_ID:
        DOCS.append(doc)
        BY_ID[doc["id"]] = doc
    show(page_images(doc["id"])[0])

**What you just did.** A PDF opened and looked at; a classical OCR that returns a table of words with its own doubts; a free OCR model on your own machine; a large model that returns a transcription and variables; three readings compared without a ground truth; and a loop that turns pages into rows. In Part 5 of the session, the same idea on 300 leaflets: a form, filled in by two models, checked against a catalogue.